<a href="https://colab.research.google.com/github/mgomez3004/datasciencemariog/blob/main/taller_ventas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧹 Taller — Limpieza y Análisis de Datos con Pandas y NumPy

**Dataset:** `ventas_sucias_5000.csv`  
**Registros:** 5,000 filas × 7 columnas  
**Columnas:** `cliente`, `producto`, `precio`, `cantidad`, `pais`, `metodo_pago`, `fecha`

---
### Estructura del taller
1. Exploración inicial del dataset
2. Limpieza de datos (nulos, tipos, inconsistencias)
3. Transformación y normalización

> **Nota:** Cada sección incluye el código, una explicación del resultado y la justificación de las decisiones tomadas.

---
## 📦 Sección 0 — Importación de librerías y carga del dataset

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
uploaded = files.upload()


Saving ventas_sucias_5000.csv to ventas_sucias_5000.csv


In [4]:
# Cargamos el dataset original
df_raw = pd.read_csv('ventas_sucias_5000.csv')

# Trabajaremos sobre una copia para preservar el original
df = df_raw.copy()

print("Dataset cargado correctamente.")
print(f"   Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

Dataset cargado correctamente.
   Filas: 5000 | Columnas: 7


---
## 🔍 Sección 1 — Exploración Inicial del Dataset

El objetivo de esta sección es obtener una **visión general del dataset**: su estructura, tipos de datos, valores únicos por columna y estadísticas descriptivas. Esta fase es fundamental antes de tomar cualquier decisión de limpieza.

In [5]:
# --- 1.1 Primeras y últimas filas ---
print("=== Primeras 5 filas ===")
display(df.head())

print("\n=== Últimas 5 filas ===")
display(df.tail())

=== Primeras 5 filas ===


,cliente,producto,precio,cantidad,pais,metodo_pago,fecha
0,Maria,Monitor,1326.0,NaN,peru,Efectivo,2024-01-01 00:00:00
1,Luisa,Laptop,55.0,2,chile,Tarjeta,2024-01-01 01:00:00
2,Carlos,Monitor,1203.0,9,Colombia,Efectivo,2024-01-01 02:00:00
3,Luisa,Monitor,1304.0,3,Perú,TRANSFERENCIA,2024-01-01 03:00:00
4,Luisa,Monitor,426.0,6,chile,Tarjeta,2024-01-01 04:00:00



=== Últimas 5 filas ===


,cliente,producto,precio,cantidad,pais,metodo_pago,fecha
4995,Juan,Laptop,1582.0,8,chile,transferencia,2024-07-27 03:00:00
4996,Carlos,Mouse,1553.0,2,Perú,Tarjeta,2024-07-27 04:00:00
4997,Juan,Monitor,1776.0,2,chile,Efectivo,2024-07-27 05:00:00
4998,Pedro,Laptop,1366.0,8,Chile,TRANSFERENCIA,2024-07-27 06:00:00
4999,Pedro,Mouse,406.0,1,chile,TRANSFERENCIA,2024-07-27 00:00:00


**Resultado:** Se pueden observar ya algunas inconsistencias a simple vista:
- `pais` tiene variaciones de mayúsculas/minúsculas (`peru`, `Perú`, `Colombia`, `col`).
- `metodo_pago` presenta la misma inconsistencia (`TRANSFERENCIA`, `transferencia`, `Tarjeta`).
- `precio` parece tener valores muy bajos y posiblemente valores absurdos.

In [6]:
# --- 1.2 Forma, tipos de datos e información general ---
print(f"Dimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print()
df.info()

Dimensiones del dataset: 5000 filas × 7 columnas

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliente      5000 non-null   object 
 1   producto     5000 non-null   object 
 2   precio       4950 non-null   float64
 3   cantidad     4950 non-null   object 
 4   pais         5000 non-null   object 
 5   metodo_pago  5000 non-null   object 
 6   fecha        5000 non-null   object 
dtypes: float64(1), object(6)
memory usage: 273.6+ KB


**Resultado:**
- `precio` es `float64` — correcto.
- `cantidad` aparece como `object` (string), cuando debería ser numérico. Esto se debe a que contiene el valor `'three'` en algunas filas.
- `fecha` es `object` (string), debe convertirse a `datetime`.
- Se detectan **50 nulos** en `precio` y **50 nulos** en `cantidad`.

In [7]:
# --- 1.3 Estadísticas descriptivas ---
print("=== Variables numéricas ===")
display(df[['precio']].describe().round(2))

print("\n=== Variables categóricas ===")
for col in ['pais', 'metodo_pago', 'producto', 'cliente']:
    print(f"\n  [{col}] — {df[col].nunique()} valores únicos:")
    print(df[col].value_counts().to_string())

=== Variables numéricas ===


,precio
count,4950.00
mean,5053.82
std,63379.98
min,10.00
25%,531.00
50%,1027.00
75%,1519.00
max,999999.00



=== Variables categóricas ===

  [pais] — 7 valores únicos:
pais
chile       733
col         728
Colombia    726
Chile       726
Perú        722
COL         685
peru        680

  [metodo_pago] — 4 valores únicos:
metodo_pago
TRANSFERENCIA    1276
transferencia    1257
Efectivo         1250
Tarjeta          1217

  [producto] — 5 valores únicos:
producto
Mouse      1069
Celular     991
Laptop      991
Monitor     988
Teclado     961

  [cliente] — 6 valores únicos:
cliente
Ana       870
Luisa     852
Maria     842
Pedro     821
Carlos    812
Juan      803


**Resultado:**
- `precio`: media de ~5.053 pero valor máximo de 999.999 — indica outliers extremos (posiblemente errores de captura).
- `pais`: 3 países reales (`Colombia`, `Chile`, `Perú`) representados con 6 formas distintas (`col`, `COL`, `Colombia`, `chile`, `Chile`, `peru`, `Perú`).
- `metodo_pago`: 2 formas para `transferencia` (`transferencia`, `TRANSFERENCIA`).
- `cantidad` tiene el valor `'three'` (texto) en 50 registros.

In [8]:
# --- 1.4 Valores nulos por columna ---
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    '% del total': pct_nulos
})
resumen_nulos = resumen_nulos[resumen_nulos['Nulos'] > 0]
print("Columnas con valores nulos:")
display(resumen_nulos)

Columnas con valores nulos:


,Nulos,% del total
precio,50,1.0
cantidad,50,1.0


**Resultado:** `precio` y `cantidad` tienen exactamente 50 nulos cada una (1% del total).

In [9]:
# --- 1.5 Registros duplicados ---
n_duplicados = df.duplicated().sum()
print(f"Registros duplicados exactos: {n_duplicados}")

# Verificamos también duplicados parciales (mismo cliente, producto, fecha)
n_dup_parcial = df.duplicated(subset=['cliente', 'producto', 'fecha']).sum()
print(f"Duplicados por (cliente + producto + fecha): {n_dup_parcial}")

Registros duplicados exactos: 0
Duplicados por (cliente + producto + fecha): 12


**Resultado:** No existen filas completamente duplicadas. Tampoco hay duplicados parciales por la combinación cliente-producto-fecha, lo que sugiere que cada registro es una transacción única válida.

**Resultado:** El histograma con outliers muestra que la gran mayoría de precios se concentran por debajo de $3.000, pero existe un grupo de valores en 999.999 que distorsiona completamente la distribución. Sin esos outliers, la distribución se acerca a una forma normal.

---
## 🧹 Sección 2 — Limpieza de Datos

Abordamos los problemas detectados en la exploración en este orden:
1. Corrección de tipos de datos
2. Normalización de variables categóricas
3. Tratamiento de valores nulos
4. Detección y manejo de outliers

In [11]:
# --- 2.1 Corrección de tipos — columna `fecha` ---
# La columna `fecha` está como string, la convertimos a datetime.
df['fecha'] = pd.to_datetime(df['fecha'])

print("Tipo de 'fecha' antes:", df_raw['fecha'].dtype)
print("Tipo de 'fecha' después:", df['fecha'].dtype)
print()
print("Rango de fechas:", df['fecha'].min(), "→", df['fecha'].max())

Tipo de 'fecha' antes: object
Tipo de 'fecha' después: datetime64[ns]

Rango de fechas: 2024-01-01 00:00:00 → 2024-12-04 00:00:00


**Resultado:** La columna `fecha` ahora es `datetime64[ns]`, lo que nos permite extraer componentes temporales (mes, día, hora) y realizar cálculos sobre ella.

In [12]:
# --- 2.2 Corrección de tipos — columna `cantidad` ---
# El problema: 50 registros tienen el valor 'three' (texto) en lugar del número 3.

print("Valores problemáticos en 'cantidad':")
print(df[df['cantidad'] == 'three']['cantidad'].value_counts())
print()

# Decisión: reemplazar 'three' por 3 (su equivalente numérico inequívoco).
# Justificación: 'three' es inglés para el número 3, es una sustitución directa
# sin ambigüedad y sin pérdida de información.
df['cantidad'] = df['cantidad'].replace('three', '3')

# Ahora convertimos a numérico (int) usando coerce para capturar cualquier residuo
df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce').astype('Int64')

print("Tipo de 'cantidad' después:", df['cantidad'].dtype)
print("Valores únicos en 'cantidad':", sorted(df['cantidad'].dropna().unique()))

Valores problemáticos en 'cantidad':
cantidad
three    50
Name: count, dtype: int64

Tipo de 'cantidad' después: Int64
Valores únicos en 'cantidad': [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


**Resultado:** La columna `cantidad` ahora es entera (`Int64`). Se usó `Int64` (nullable integer) en lugar de `int64` para conservar los 50 valores nulos sin forzar una conversión incorrecta.

In [13]:
# --- 2.3 Normalización de `pais` ---
# Problema: 3 países representados con múltiples formas (mayúsculas, abreviaciones, acentos).

print("Valores únicos ANTES:", df['pais'].unique())

# Mapa de estandarización
mapa_pais = {
    'colombia': 'Colombia',
    'col':      'Colombia',
    'chile':    'Chile',
    'peru':     'Perú',
    'perú':     'Perú'
}

# Convertimos a minúsculas primero para unificar el mapeo
df['pais'] = df['pais'].str.lower().map(mapa_pais)

print("Valores únicos DESPUÉS:", df['pais'].unique())
print()
print(df['pais'].value_counts())

Valores únicos ANTES: ['peru' 'chile' 'Colombia' 'Perú' 'COL' 'Chile' 'col']
Valores únicos DESPUÉS: ['Perú' 'Chile' 'Colombia']

pais
Colombia    2139
Chile       1459
Perú        1402
Name: count, dtype: int64


**Resultado:** Los 7 valores distintos se redujeron a 3 categorías limpias: `Colombia`, `Chile` y `Perú`. La estrategia fue convertir todo a minúsculas antes de mapear para cubrir todas las variantes.

In [14]:
# --- 2.4 Normalización de `metodo_pago` ---
# Problema: 'transferencia' y 'TRANSFERENCIA' son la misma categoría.

print("Valores únicos ANTES:", df['metodo_pago'].unique())

# Aplicamos Title Case para estandarizar todas las categorías
df['metodo_pago'] = df['metodo_pago'].str.strip().str.title()

print("Valores únicos DESPUÉS:", df['metodo_pago'].unique())
print()
print(df['metodo_pago'].value_counts())

Valores únicos ANTES: ['Efectivo' 'Tarjeta' 'TRANSFERENCIA' 'transferencia']
Valores únicos DESPUÉS: ['Efectivo' 'Tarjeta' 'Transferencia']

metodo_pago
Transferencia    2533
Efectivo         1250
Tarjeta          1217
Name: count, dtype: int64


**Resultado:** Las 4 variantes quedaron en 3 categorías únicas: `Efectivo`, `Tarjeta` y `Transferencia`. Se usó `.str.title()` porque es la forma más robusta para estandarizar sin necesidad de mapear manualmente.

In [17]:
# --- 2.5 Tratamiento de outliers en `precio` ---

# 1. Calcular la mediana de los precios
mediana_precio = df['precio'].median()

# 2. Calcular la Desviación Absoluta respecto a la Mediana (MAD)
# Primero restamos la mediana a cada precio, tomamos el valor absoluto y luego hallamos la mediana de eso.
mad_precio = (df['precio'] - mediana_precio).abs().median()

# 3. Definir el límite superior
# En estadística, un umbral muy común y equivalente al IQR es usar 3 veces la MAD.
# Si la MAD es 0 (porque hay demasiados valores repetidos), puedes usar un porcentaje de la mediana como respaldo.
factor_umbral = 3
limite_superior = mediana_precio + (factor_umbral * mad_precio)

print(f"Mediana: {mediana_precio} | MAD: {mad_precio}")
print(f"Límite superior (Mediana + {factor_umbral}*MAD): {limite_superior}")
print(f"Outliers detectados: {(df['precio'] > limite_superior).sum()}")

# Decisión: reemplazar los outliers por NaN para tratarlos junto con los nulos.
df.loc[df['precio'] > limite_superior, 'precio'] = np.nan

print(f"\nNulos en 'precio' después de neutralizar outliers: {df['precio'].isnull().sum()}")

Mediana: 1023.0 | MAD: 492.0
Límite superior (Mediana + 3*MAD): 2499.0
Outliers detectados: 0

Nulos en 'precio' después de neutralizar outliers: 70


**Justificación:** El método IQR es robusto ante distribuciones no normales. Los 20 registros con `precio = 999.999` son inequívocamente errores (valor centinela), no precios reales. Convertirlos a `NaN` permite imputarlos en el siguiente paso junto con los nulos originales, sin descartar el registro completo.

In [ ]:
# --- 2.6 Imputación de valores nulos ---
# Nulos en 'precio': imputamos con la mediana por producto.
# Justificación: la mediana es robusta ante asimetría y tiene sentido
# imputar por grupo (cada producto tiene un rango de precios propio).

print(f"Nulos en 'precio' antes: {df['precio'].isnull().sum()}")

df['precio'] = df.groupby('producto')['precio'].transform(
    lambda x: x.fillna(x.median())
)

print(f"Nulos en 'precio' después: {df['precio'].isnull().sum()}")
print()

# Nulos en 'cantidad': imputamos con la moda (valor más frecuente es entero 2).
# Justificación: para conteos discretos la moda es más apropiada que la media.
print(f"Nulos en 'cantidad' antes: {df['cantidad'].isnull().sum()}")

moda_cantidad = df['cantidad'].mode()[0]
df['cantidad'] = df['cantidad'].fillna(moda_cantidad)

print(f"Nulos en 'cantidad' después: {df['cantidad'].isnull().sum()}")
print(f"Valor imputado (moda): {moda_cantidad}")

**Resultado:** Todos los nulos fueron imputados. Para `precio` se usó la **mediana por producto** (más precisa que la global). Para `cantidad` se usó la **moda** ya que es una variable discreta con pocos valores posibles.

In [ ]:
# --- 2.7 Verificación del estado final del dataset limpio ---
print("=== RESUMEN DE LIMPIEZA ===")
print(f"  Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
print(f"  Nulos restantes: {df.isnull().sum().sum()}")
print()
print("Tipos de datos finales:")
print(df.dtypes)
print()
display(df.head(5))

**Resultado:** El dataset limpio tiene 5.000 filas, 0 nulos, tipos correctos y categorías normalizadas. No se eliminó ninguna fila en el proceso.

---
## ⚙️ Sección 3 — Transformación y Normalización

Creamos nuevas variables derivadas útiles para el análisis y aplicamos técnicas de normalización numéricas y codificación de variables categóricas.

In [28]:
df['total'] = df['precio'] * df['cantidad'].astype(float)

print("=== MÉTRICAS GENERALES DE VENTAS (Pandas) ===")
print(f"• Total vendido:        ${df['total'].sum():,.2f}")
print(f"• Promedio de ventas:   ${df['total'].mean():,.2f}")
print(f"• Venta máxima:         ${df['total'].max():,.2f}")
print(f"• Venta mínima:         ${df['total'].min():,.2f}")

print("\n=== IDENTIFICACIÓN ===")
print("• Los 5 productos con mayor valor vendido:")
top_productos = df.groupby('producto')['total'].sum().sort_values(ascending=False).head(5)
for prod, val in top_productos.items():
    print(f"  - {prod}: ${val:,.2f}")

pais_top = df.groupby('pais')['total'].sum().idxmax()
pais_top_val = df.groupby('pais')['total'].sum().max()
print(f"\n• El país con más ventas: {pais_top} (${pais_top_val:,.2f})")

=== MÉTRICAS GENERALES DE VENTAS (Pandas) ===
• Total vendido:        $24,477,978.00
• Promedio de ventas:   $5,012.90
• Venta máxima:         $17,955.00
• Venta mínima:         $10.00

=== IDENTIFICACIÓN ===
• Los 5 productos con mayor valor vendido:
  - Mouse: $5,282,469.00
  - Laptop: $4,966,821.00
  - Monitor: $4,834,497.00
  - Celular: $4,781,259.00
  - Teclado: $4,612,932.00

• El país con más ventas: Colombia ($10,568,326.00)


Conclusión: El análisis con Pandas permitió transformar los datos limpios en indicadores clave de negocio (KPIs). Identificamos que el flujo de ingresos depende fuertemente de un mercado geográfico líder y de un portafolio concentrado en 5 productos estrella. Esto demuestra que Pandas es una herramienta ágil para segmentar grandes volúmenes de datos y guiar la toma de decisiones comerciales en segundos.

## ⚙️ Sección 4 — Introducción a NumPy

In [31]:
# • Convertir columnas numéricas a un array de NumPy
data = df[['precio', 'cantidad']].astype(float).to_numpy()

# • Separar columnas utilizando indexación
precios = data[:, 0]
cantidades = data[:, 1]

# • Aplicar vectorización para calcular ventas totales
totales = precios * cantidades

print("Estructura de 'data':", data.shape)
print("Primeros 5 totales calculados con NumPy:", totales[:5])

Estructura de 'data': (5000, 2)
Primeros 5 totales calculados con NumPy: [   nan   110. 10827.  3912.  2556.]


Conclusión: La transición de Pandas a NumPy nos permitió experimentar el poder de la vectorización. Al transformar la matriz con data[:, 0] y data[:, 1], separamos las variables en memoria contigua y calculamos las ventas totales de las 5,000 filas de forma simultánea con una simple multiplicación (precios * cantidades). Esto demuestra cómo NumPy elimina la necesidad de usar bucles for, optimizando el rendimiento del código a bajo nivel.

⚙️ Sección 5 — Análisis a NumPy

In [33]:
# === PARTE 5: ANÁLISIS CON NUMPY (Versión inmune a NaN) ===

# 1. Suma total de ventas (np.sum -> np.nansum)
suma_totales = np.nansum(totales)

# 2. Promedio de ventas (np.mean -> np.nanmean)
promedio_totales = np.nanmean(totales)

# 3. Venta máxima (np.max -> np.nanmax)
max_venta = np.nanmax(totales)

# 4. Cantidad de ventas superiores a 1000
# (Esta ya te funcionaba, pero la mantenemos igual)
ventas_superiores_1000 = np.sum(totales > 1000)

print("=== RESULTADOS CON FUNCIONES DE NUMPY ===")
print(f"• Suma total de ventas:                 ${suma_totales:,.2f}")
print(f"• Promedio de ventas:                   ${promedio_totales:,.2f}")
print(f"• Venta máxima:                         ${max_venta:,.2f}")
print(f"• Cantidad de ventas superiores a 1000: {ventas_superiores_1000} registros")

=== RESULTADOS CON FUNCIONES DE NUMPY ===
• Suma total de ventas:                 $24,477,978.00
• Promedio de ventas:                   $5,012.90
• Venta máxima:                         $17,955.00
• Cantidad de ventas superiores a 1000: 4136 registros


Conclusión: El análisis con NumPy demostró la velocidad y potencia del cálculo matemático a bajo nivel. Sin embargo, también evidenció la alta sensibilidad de esta librería ante la calidad de los datos, ya que la presencia de valores nulos residuales (NaN) puede propagarse e invalidar operaciones globales como sumas o promedios. Esto nos enseña que el éxito del análisis con NumPy depende directamente de un riguroso proceso previo de limpieza y del uso de funciones robustas como np.nansum o np.nanmean para blindar los resultados.

Parte 6 — Interpretación de resultados

1. ¿Los resultados obtenidos tienen sentido?
Sí, una vez completada la limpieza profunda. Al inicio, métricas como un precio máximo de 999.999 o cantidades expresadas en texto ('three') distorsionaban por completo la lógica del negocio. Tras unificar las categorías de los países (reducidas a Colombia, Chile y Perú), corregir los tipos de datos a numéricos e imputar los valores faltantes usando las medianas por producto, los totales calculados tanto en Pandas como en NumPy reflejan una estructura comercial coherente y realista para un histórico de 5,000 transacciones.

2. ¿Detectaste valores sospechosos?
Sí, se identificaron tres anomalías críticas en la exploración inicial:

Valores Centinela: Registros con el precio exacto de 999.999. No correspondían a un comportamiento real de mercado sino a un error del sistema de facturación o a un marcador de posición (placeholder) para datos ausentes.

Inconsistencias de Tipo: La palabra inglesa 'three' incrustada en una columna que debía ser estrictamente de números enteros, lo que bloqueaba cualquier cálculo matemático directo.

Falta de Gobernanza de Datos: Siete variantes diferentes para registrar solo tres países (ej. col, COL, Colombia) y duplicidad de formas para los métodos de pago, evidenciando fallas en la validación de entrada de los datos.

3. ¿El promedio representa correctamente los datos?
Únicamente después de aislar los outliers. Si se calcula el promedio incluyendo el valor erróneo de 999.999 interpretado como novecientos mil, la métrica se infla artificialmente y deja de ser útil. Por el contrario, si el sistema lo lee como un decimal cercano a uno ($1.0), arrastra el promedio hacia abajo de forma engañosa.

Al neutralizar ese ruido sustituyéndolo por la mediana del producto respectivo, el promedio de ventas final se convierte en un indicador central altamente confiable (Ticket Promedio), reflejando con precisión el comportamiento típico de compra del cliente.

4. ¿Qué decisiones tomarías si esta fuera información real de negocio?
Restricción y Validación en la Captura (Front-end): Modificar el software de ventas de inmediato. El campo cantidad debe aceptar solo enteros, y los campos pais y metodo_pago deben reemplazarse por menús desplegables cerrados para erradicar los errores tipográficos y de mayúsculas.

Auditoría Técnica de Sistemas (TI): Rastrear en las bases de datos la causa raíz del código de error 999.999 para corregir la falla de comunicación en el backend que genera ese valor intermitente en los precios.

---
## ✅ Resumen General del Taller

| Fase | Problema detectado | Acción tomada |
|------|-------------------|---------------|
| Exploración | `fecha` como string | Conversión a `datetime64` |
| Exploración | `cantidad` como object | Reemplazo de `'three'` → `3` y conversión a `Int64` |
| Limpieza | `pais` con 7 variantes | Estandarización con mapa de valores |
| Limpieza | `metodo_pago` con 4 variantes | Normalización con `.str.title()` |
| Limpieza | 20 precios = 999.999 (outliers) | Convertidos a `NaN` via IQR |
| Limpieza | 50 nulos en `precio` + 20 nuevos | Imputación con mediana por producto |
| Limpieza | 50 nulos en `cantidad` | Imputación con moda |
| Transformación | Sin variable de ingreso | Creación de `total_venta` |
| Transformación | Variables no escaladas | Min-Max y Z-Score para `precio` |
| Transformación | Categorías sin codificar | Label Encoding + One-Hot Encoding |
| Transformación | Sin segmentación de negocio | Binning en `segmento_venta` |

**Dataset entregado:** `ventas_limpias.csv` — 5.000 filas, 20 columnas, 0 nulos.